In [ ]:
from pathlib import Path
from google.colab import drive

# Connect Google Drive to this Colab session.
drive.mount("/content/drive")

# Define the main folder for the non-imaging preprocessing workflow.
PROJECT_DIR = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging"
)

# Confirm that the expected project folder exists.
if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"The expected project folder was not found:\n{PROJECT_DIR}"
    )

print(f"Project folder connected successfully:\n{PROJECT_DIR}\n")

# Display the folders currently present inside the project directory.
print("Current project contents:")
for path in sorted(PROJECT_DIR.iterdir()):
    item_type = "folder" if path.is_dir() else "file"
    print(f"- {path.name} ({item_type})")

# 1. Demographics preprocessing

define the same project paths used in the main ADNI non-imaging notebook.

The raw PTDEMOG file will only be read from the `raw` folder. Cleaned longitudinal data will be saved in `interim`, final participant-level demographic data will eventually be saved in `processed`, and all PTDEMOG quality-control outputs will be saved in `qc`.

In [ ]:
from pathlib import Path

# Define the main non-imaging project folder.
PROJECT_DIR = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging"
)

# Define the folders used throughout the preprocessing workflow.
RAW_DIR = PROJECT_DIR / "raw"
INVENTORY_DIR = PROJECT_DIR / "inventory"
QC_DIR = PROJECT_DIR / "qc"
INTERIM_DIR = PROJECT_DIR / "interim"
PROCESSED_DIR = PROJECT_DIR / "processed"
MANIFESTS_DIR = PROJECT_DIR / "manifests"

PROJECT_FOLDERS = {
    "raw": RAW_DIR,
    "inventory": INVENTORY_DIR,
    "qc": QC_DIR,
    "interim": INTERIM_DIR,
    "processed": PROCESSED_DIR,
    "manifests": MANIFESTS_DIR,
}

# Confirm that every expected project folder exists.
missing_folders = [
    str(folder_path)
    for folder_path in PROJECT_FOLDERS.values()
    if not folder_path.exists()
]

if missing_folders:
    raise FileNotFoundError(
        "The following expected project folders were not found:\n"
        + "\n".join(missing_folders)
    )

print("Project folders validated successfully.\n")

for folder_name, folder_path in PROJECT_FOLDERS.items():
    files_inside = [
        path
        for path in folder_path.rglob("*")
        if path.is_file()
    ]

    print(
        f"{folder_name:<10} | "
        f"{len(files_inside):>4} file(s) | "
        f"{folder_path}"
    )

# 2. Import the libraries required for PTDEMOG preprocessing

import only the libraries needed for loading, cleaning, validating, and summarising the PTDEMOG table.

In [ ]:
import numpy as np
import pandas as pd

# 3. Load the raw PTDEMOG table

load the preserved PTDEMOG file directly from the `Cohort, dates and source-of-truth tables` folder.

The raw source file will remain unchanged. This notebook will create separate working copies and processed outputs for all cleaning and quality-control steps.

In [ ]:
# Define the exact path to the preserved raw PTDEMOG file.
PTDEMOG_PATH = (
    RAW_DIR
    / "Cohort, dates and source-of-truth tables"
    / "All_Subjects_PTDEMOG_11Jul2026.csv"
)

# Confirm that the expected source file exists.
if not PTDEMOG_PATH.exists():
    raise FileNotFoundError(
        f"The PTDEMOG source file was not found:\n{PTDEMOG_PATH}"
    )

# Load the raw PTDEMOG table without modifying the source file.
ptdemog_raw = pd.read_csv(
    PTDEMOG_PATH,
    low_memory=False,
)

print("PTDEMOG loaded successfully.")
print(f"Source file:\n{PTDEMOG_PATH}\n")

print(f"Rows: {len(ptdemog_raw):,}")
print(f"Columns: {ptdemog_raw.shape[1]:,}")
print(
    f"Unique participants: "
    f"{ptdemog_raw['RID'].nunique(dropna=True):,}"
)

display(ptdemog_raw.head())

# 4. Define the PTDEMOG output paths

define the filenames that will be used for the cleaned longitudinal PTDEMOG table and its quality-control outputs.

The cleaned longitudinal output will preserve all usable records. A single participant-level demographic record will not be created until the shared reference-date and record-selection rules have been established.

In [ ]:
# Define the planned PTDEMOG output files.
PTDEMOG_LONGITUDINAL_OUTPUT = (
    INTERIM_DIR
    / "ptdemog_cleaned_longitudinal.csv"
)

PTDEMOG_MISSINGNESS_OVERALL_OUTPUT = (
    QC_DIR
    / "ptdemog_missingness_overall.csv"
)

PTDEMOG_MISSINGNESS_BY_PHASE_OUTPUT = (
    QC_DIR
    / "ptdemog_missingness_by_phase.csv"
)

PTDEMOG_PARTICIPANT_CONSISTENCY_OUTPUT = (
    QC_DIR
    / "ptdemog_participant_consistency.csv"
)

PTDEMOG_CORE_CONFLICTS_OUTPUT = (
    QC_DIR
    / "ptdemog_core_conflicts.csv"
)

print("PTDEMOG output paths defined successfully.\n")

print(
    "Cleaned longitudinal table:\n"
    f"{PTDEMOG_LONGITUDINAL_OUTPUT}\n"
)

print("Quality-control outputs:")
print(f"- {PTDEMOG_MISSINGNESS_OVERALL_OUTPUT}")
print(f"- {PTDEMOG_MISSINGNESS_BY_PHASE_OUTPUT}")
print(f"- {PTDEMOG_PARTICIPANT_CONSISTENCY_OUTPUT}")
print(f"- {PTDEMOG_CORE_CONFLICTS_OUTPUT}")

# 5. Create the focused PTDEMOG working table

create a separate working copy containing only the variables needed for demographic modelling, optional ablation, fairness analysis, temporal alignment, quality control, and leakage review.

preserve all longitudinal PTDEMOG records at this stage. No participant-level record selection, baseline alignment, age calculation, or merging with other modalities will be performed yet.

In [ ]:
# Define the PTDEMOG variables according to their intended role.

processing_columns = [
    "PHASE",
    "PTID",
    "RID",
    "VISCODE",
    "VISCODE2",
    "VISDATE",
]

age_source_columns = [
    "PTDOB",
    "PTDOBYY",
]

core_feature_columns = [
    "PTGENDER",
    "PTEDUCAT",
    "PTTLANG",
]

optional_feature_columns = [
    "PTPLANG",
    "PTMARRY",
    "PTHAND",
    "PTNLANG",
    "PTLANGTTL",
    "PTENGSPK",
    "PTENGSPKAGE",
    "PTWORK",
    "PTNOTRT",
    "PTRTYR",
]

fairness_columns = [
    "PTRACCAT",
    "PTETHCAT",
    "PTETHCATH",
    "PTASIAN",
    "PTOPI",
]

leakage_review_columns = [
    "PTADDX",
    "PTADBEG",
    "PTCOGBEG",
]

qc_columns = [
    "PTSOURCE",
    "SITEID",
    "ID",
    "DD_CRF_VERSION_LABEL",
    "LANGUAGE_CODE",
    "HAS_QC_ERROR",
    "USERDATE",
    "USERDATE2",
    "update_stamp",
]

# Combine the selected columns while preserving their intended order.
selected_columns = (
    processing_columns
    + age_source_columns
    + core_feature_columns
    + optional_feature_columns
    + fairness_columns
    + leakage_review_columns
    + qc_columns
)

# Create an independent working copy for PTDEMOG preprocessing.
ptdemog_working = ptdemog_raw[selected_columns].copy()

print(f"Rows in PTDEMOG working table: {len(ptdemog_working):,}")
print(f"Columns retained: {ptdemog_working.shape[1]}")

display(ptdemog_working.head())

# 6. Standardise identifiers, visit codes, and dates

clean the participant identifiers, standardise the visit-code fields, and parse the PTDEMOG date columns.

`VISDATE` will be treated as the participant-related date for this table. Administrative fields such as `USERDATE`, `USERDATE2`, and `update_stamp` will be parsed separately and will not be used as clinical dates.

also extract the participant's birth year from `PTDOB` and `PTDOBYY`. not calculate age yet because the common reference-date rule has not been established.

In [ ]:
# Standardise the participant identifiers.
ptdemog_working["RID_CLEAN"] = pd.to_numeric(
    ptdemog_working["RID"],
    errors="coerce",
).astype("Int64")

ptdemog_working["PTID_CLEAN"] = (
    ptdemog_working["PTID"]
    .astype("string")
    .str.strip()
)

# Standardise the phase and visit-code fields.
ptdemog_working["PHASE_CLEAN"] = (
    ptdemog_working["PHASE"]
    .astype("string")
    .str.strip()
    .str.upper()
)

for column in ["VISCODE", "VISCODE2"]:
    ptdemog_working[f"{column}_CLEAN"] = (
        ptdemog_working[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# Parse the participant-related PTDEMOG date.
ptdemog_working["VISDATE_PARSED"] = pd.to_datetime(
    ptdemog_working["VISDATE"],
    errors="coerce",
)

# Parse administrative dates separately.
for column in ["USERDATE", "USERDATE2", "update_stamp"]:
    ptdemog_working[f"{column}_PARSED"] = pd.to_datetime(
        ptdemog_working[column],
        errors="coerce",
    )

# Extract the year from PTDOB, which is commonly stored as month/year.
ptdemog_working["PTDOB_YEAR"] = pd.to_numeric(
    ptdemog_working["PTDOB"]
    .astype("string")
    .str.extract(r"(\d{4})", expand=False),
    errors="coerce",
).astype("Int64")

# Extract the year from the standardised PTDOBYY date field.
ptdemog_working["PTDOBYY_PARSED"] = pd.to_datetime(
    ptdemog_working["PTDOBYY"],
    errors="coerce",
)

ptdemog_working["PTDOBYY_YEAR"] = (
    ptdemog_working["PTDOBYY_PARSED"]
    .dt.year
    .astype("Int64")
)

# Flag records where the two available birth-year sources disagree.
ptdemog_working["BIRTH_YEAR_CONFLICT"] = (
    ptdemog_working["PTDOB_YEAR"].notna()
    & ptdemog_working["PTDOBYY_YEAR"].notna()
    & (
        ptdemog_working["PTDOB_YEAR"]
        != ptdemog_working["PTDOBYY_YEAR"]
    )
)

print("Identifier and date standardisation completed.\n")

print(
    "Rows with missing cleaned RID:",
    f"{ptdemog_working['RID_CLEAN'].isna().sum():,}",
)

print(
    "Rows with missing parsed VISDATE:",
    f"{ptdemog_working['VISDATE_PARSED'].isna().sum():,}",
)

print(
    "Rows with conflicting PTDOB and PTDOBYY years:",
    f"{ptdemog_working['BIRTH_YEAR_CONFLICT'].sum():,}",
)

display(
    ptdemog_working[
        [
            "RID_CLEAN",
            "PTID_CLEAN",
            "PHASE_CLEAN",
            "VISCODE_CLEAN",
            "VISCODE2_CLEAN",
            "VISDATE_PARSED",
            "PTDOB",
            "PTDOBYY",
            "PTDOB_YEAR",
            "PTDOBYY_YEAR",
            "BIRTH_YEAR_CONFLICT",
        ]
    ].head()
)

# 7. Inspect PTDEMOG records with missing visit dates

I found eight PTDEMOG records for which `VISDATE` could not be parsed.

inspect these rows together with their participant identifiers, visit codes, phase, administrative dates, QC flag, and core demographic values. also check whether the affected participants have other PTDEMOG records with valid visit dates.

not remove these rows yet. A missing date on one record should not cause the participant to be excluded if another usable demographic record is available.

In [ ]:
# Select the PTDEMOG rows with no usable visit date.
ptdemog_missing_visit_date = (
    ptdemog_working[
        ptdemog_working["VISDATE_PARSED"].isna()
    ]
    .copy()
)

# Identify all participants affected by at least one missing VISDATE row.
affected_rids = (
    ptdemog_missing_visit_date["RID_CLEAN"]
    .dropna()
    .unique()
)

# Retrieve the complete PTDEMOG history for the affected participants.
ptdemog_missing_date_participant_history = (
    ptdemog_working[
        ptdemog_working["RID_CLEAN"].isin(affected_rids)
    ][
        [
            "RID_CLEAN",
            "PTID_CLEAN",
            "PHASE_CLEAN",
            "VISCODE_CLEAN",
            "VISCODE2_CLEAN",
            "VISDATE",
            "VISDATE_PARSED",
            "USERDATE_PARSED",
            "USERDATE2_PARSED",
            "update_stamp_PARSED",
            "HAS_QC_ERROR",
            "PTGENDER",
            "PTEDUCAT",
            "PTTLANG",
            "PTPLANG",
            "PTDOB_YEAR",
        ]
    ]
    .sort_values(
        by=[
            "RID_CLEAN",
            "VISDATE_PARSED",
            "USERDATE_PARSED",
        ],
        na_position="last",
    )
)

# Summarise whether each affected participant has another valid dated record.
ptdemog_missing_date_summary = (
    ptdemog_working[
        ptdemog_working["RID_CLEAN"].isin(affected_rids)
    ]
    .groupby("RID_CLEAN")
    .agg(
        PTID=("PTID_CLEAN", "first"),
        TOTAL_PTDEMOG_ROWS=("RID_CLEAN", "size"),
        VALID_DATED_ROWS=("VISDATE_PARSED", lambda x: x.notna().sum()),
        MISSING_DATED_ROWS=("VISDATE_PARSED", lambda x: x.isna().sum()),
    )
    .reset_index()
)

print(
    "Rows with missing VISDATE:",
    len(ptdemog_missing_visit_date),
)

print(
    "Participants affected:",
    len(affected_rids),
)

print(
    "Affected participants with no alternative dated PTDEMOG record:",
    (
        ptdemog_missing_date_summary["VALID_DATED_ROWS"] == 0
    ).sum(),
)

display(ptdemog_missing_date_summary)
display(ptdemog_missing_date_participant_history)

# 8. Flag PTDEMOG records with missing visit dates

classify PTDEMOG rows according to whether they have a valid participant-related visit date and whether another dated PTDEMOG record exists for the same participant.

Administrative dates will not be substituted for `VISDATE`, because they represent database entry or modification rather than the date of the demographic assessment.

Undated records will be preserved for provenance and quality control. They will not be used in later date-based record selection unless another source establishes a defensible assessment date.

In [ ]:
# Count the number of valid dated PTDEMOG records available for each participant.
valid_dated_rows_by_rid = (
    ptdemog_working
    .groupby("RID_CLEAN")["VISDATE_PARSED"]
    .transform(lambda series: series.notna().sum())
)

# Flag whether the current row has a valid participant-related date.
ptdemog_working["HAS_VALID_VISDATE"] = (
    ptdemog_working["VISDATE_PARSED"].notna()
)

# Flag whether the participant has at least one other usable dated PTDEMOG row.
ptdemog_working["HAS_DATED_PTDEMOG_ALTERNATIVE"] = (
    valid_dated_rows_by_rid > 0
)

# Count how many of the selected demographic fields are populated on each row.
demographic_content_columns = [
    "PTDOB",
    "PTDOBYY",
    "PTGENDER",
    "PTEDUCAT",
    "PTTLANG",
    "PTPLANG",
    "PTMARRY",
    "PTHAND",
    "PTRACCAT",
    "PTETHCAT",
]

ptdemog_working["DEMOGRAPHIC_NON_MISSING_COUNT"] = (
    ptdemog_working[demographic_content_columns]
    .notna()
    .sum(axis=1)
)

# Assign a transparent temporal-use status to every PTDEMOG row.
ptdemog_working["PTDEMOG_DATE_STATUS"] = np.select(
    [
        ptdemog_working["HAS_VALID_VISDATE"],
        (
            ~ptdemog_working["HAS_VALID_VISDATE"]
            & ptdemog_working["HAS_DATED_PTDEMOG_ALTERNATIVE"]
        ),
        (
            ~ptdemog_working["HAS_VALID_VISDATE"]
            & ~ptdemog_working["HAS_DATED_PTDEMOG_ALTERNATIVE"]
            & (
                ptdemog_working["DEMOGRAPHIC_NON_MISSING_COUNT"] > 0
            )
        ),
    ],
    [
        "dated_record",
        "undated_with_dated_alternative",
        "undated_only_record_with_demographic_data",
    ],
    default="undated_only_record_without_demographic_data",
)

# Summarise the resulting categories.
ptdemog_date_status_summary = (
    ptdemog_working["PTDEMOG_DATE_STATUS"]
    .value_counts(dropna=False)
    .rename_axis("PTDEMOG_DATE_STATUS")
    .reset_index(name="ROW_COUNT")
)

display(ptdemog_date_status_summary)

# Display the undated records with their final flags.
display(
    ptdemog_working.loc[
        ~ptdemog_working["HAS_VALID_VISDATE"],
        [
            "RID_CLEAN",
            "PTID_CLEAN",
            "PHASE_CLEAN",
            "VISCODE2_CLEAN",
            "VISDATE",
            "USERDATE_PARSED",
            "USERDATE2_PARSED",
            "HAS_DATED_PTDEMOG_ALTERNATIVE",
            "DEMOGRAPHIC_NON_MISSING_COUNT",
            "PTDEMOG_DATE_STATUS",
            "PTGENDER",
            "PTEDUCAT",
            "PTTLANG",
            "PTPLANG",
            "PTDOB_YEAR",
        ],
    ]
)

# 9. Inspect categorical codes and special values

inspect the values used in the retained PTDEMOG demographic variables before cleaning them.

This will show the frequency of each category and identify placeholder values such as `-4`, `9999`, or other unusual codes. not replace or decode any values until their meanings have been confirmed.

In [ ]:
# Define the categorical and coded variables that require inspection.
coded_demographic_columns = [
    "PTSOURCE",
    "PTGENDER",
    "PTHAND",
    "PTMARRY",
    "PTTLANG",
    "PTPLANG",
    "PTENGSPK",
    "PTNLANG",
    "PTRACCAT",
    "PTETHCAT",
    "PTETHCATH",
    "PTASIAN",
    "PTOPI",
    "PTWORK",
    "PTNOTRT",
]

coded_demographic_columns = [
    column
    for column in coded_demographic_columns
    if column in ptdemog_working.columns
]

# Create a long-format frequency table for all coded variables.
coded_value_counts = []

for column in coded_demographic_columns:
    counts = (
        ptdemog_working[column]
        .value_counts(dropna=False)
        .rename_axis("RAW_VALUE")
        .reset_index(name="ROW_COUNT")
    )

    counts.insert(0, "COLUMN_NAME", column)

    counts["PERCENT_OF_ROWS"] = (
        counts["ROW_COUNT"]
        / len(ptdemog_working)
        * 100
    )

    coded_value_counts.append(counts)

ptdemog_coded_value_counts = pd.concat(
    coded_value_counts,
    ignore_index=True,
)

for column in coded_demographic_columns:
    print(f"\n{column}")
    display(
        ptdemog_coded_value_counts[
            ptdemog_coded_value_counts["COLUMN_NAME"] == column
        ].reset_index(drop=True)
    )

# 10. Load the ADNI data dictionary

load the ADNI data dictionary from the same raw-data folder as PTDEMOG.

The dictionary will be used to confirm the meaning, coding, value ranges, and special missing-value conventions for the retained PTDEMOG variables. not decode categories based only on their numerical values or frequency distributions.

In [ ]:
# Define the path to the ADNI data dictionary.
DATADIC_PATH = (
    RAW_DIR
    / "Cohort, dates and source-of-truth tables"
    / "DATADIC_11Jul2026.csv"
)

# Load the dictionary without modifying the original file.
datadic_raw = pd.read_csv(
    DATADIC_PATH,
    low_memory=False,
)

print("ADNI data dictionary loaded successfully.")
print(f"Rows: {len(datadic_raw):,}")
print(f"Columns: {datadic_raw.shape[1]:,}")

print("\nDictionary columns:")
print(datadic_raw.columns.tolist())

display(datadic_raw.head())

# 11. Extract PTDEMOG definitions from the ADNI data dictionary

extract the dictionary entries for the PTDEMOG variables retained in the working table.

Because the same variable may have different definitions or coding across ADNI phases and case-report-form versions, preserve the phase, form version, code changes, and mapping notes rather than collapsing the dictionary immediately.

In [ ]:
# Extract all PTDEMOG dictionary rows for variables retained in the working table.
ptdemog_dictionary = (
    datadic_raw[
        datadic_raw["TBLNAME"]
        .astype("string")
        .str.strip()
        .str.upper()
        .eq("PTDEMOG")
        &
        datadic_raw["FLDNAME"].isin(ptdemog_working.columns)
    ]
    [
        [
            "PHASE",
            "CRFNAME",
            "TBLNAME",
            "FLDNAME",
            "TEXT",
            "TYPE",
            "LENGTH",
            "DD_CRF_VERSION",
            "CODE",
            "UNITS",
            "STATUS",
            "CODE_CHANGES",
            "MAPPING_NOTES",
        ]
    ]
    .sort_values(
        by=[
            "FLDNAME",
            "PHASE",
            "DD_CRF_VERSION",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

print(f"PTDEMOG dictionary rows found: {len(ptdemog_dictionary):,}")
print(
    "PTDEMOG variables represented:",
    ptdemog_dictionary["FLDNAME"].nunique(),
)

for variable in sorted(ptdemog_dictionary["FLDNAME"].dropna().unique()):
    print(f"\n{'=' * 100}")
    print(variable)
    print("=" * 100)

    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.width", None,
    ):
        display(
            ptdemog_dictionary[
                ptdemog_dictionary["FLDNAME"] == variable
            ].reset_index(drop=True)
        )

# 12. Clean the core demographic variables

create cleaned versions of the main demographic variables using the definitions confirmed in the ADNI data dictionary.

The original raw columns will remain unchanged. Special values such as `-4` will be converted to missing only in the new cleaned columns.

also validate education against its documented range of 0 to 20 years.

In [ ]:
# Create cleaned numeric versions of the main PTDEMOG variables.
ptdemog_working["PTGENDER_CLEAN"] = pd.to_numeric(
    ptdemog_working["PTGENDER"],
    errors="coerce",
).replace(-4, pd.NA).astype("Int64")

ptdemog_working["PTEDUCAT_CLEAN"] = pd.to_numeric(
    ptdemog_working["PTEDUCAT"],
    errors="coerce",
)

ptdemog_working["PTTLANG_CLEAN"] = pd.to_numeric(
    ptdemog_working["PTTLANG"],
    errors="coerce",
).replace(-4, pd.NA).astype("Int64")

ptdemog_working["PTPLANG_CLEAN"] = pd.to_numeric(
    ptdemog_working["PTPLANG"],
    errors="coerce",
).replace(-4, pd.NA).astype("Int64")

# Flag education values outside the documented range.
ptdemog_working["PTEDUCAT_OUT_OF_RANGE"] = (
    ptdemog_working["PTEDUCAT_CLEAN"].notna()
    & ~ptdemog_working["PTEDUCAT_CLEAN"].between(0, 20)
)

# Replace invalid education values with missing.
ptdemog_working.loc[
    ptdemog_working["PTEDUCAT_OUT_OF_RANGE"],
    "PTEDUCAT_CLEAN",
] = pd.NA

print(
    "Education values outside the documented range:",
    int(ptdemog_working["PTEDUCAT_OUT_OF_RANGE"].sum()),
)

display(
    ptdemog_working[
        [
            "PTGENDER",
            "PTGENDER_CLEAN",
            "PTEDUCAT",
            "PTEDUCAT_CLEAN",
            "PTEDUCAT_OUT_OF_RANGE",
            "PTTLANG",
            "PTTLANG_CLEAN",
            "PTPLANG",
            "PTPLANG_CLEAN",
        ]
    ].head()
)

# 13. Inspect education values outside the valid range

I found 61 `PTEDUCAT` values outside the documented range of 0 to 20 years.

inspect the original values and their frequencies to determine whether they are special missing-value codes or genuinely implausible entries.

In [ ]:
pteducat_out_of_range_by_phase = (
    ptdemog_working.loc[
        ptdemog_working["PTEDUCAT_OUT_OF_RANGE"],
        ["PHASE_CLEAN", "PTEDUCAT"],
    ]
    .value_counts(dropna=False)
    .rename("ROW_COUNT")
    .reset_index()
    .sort_values(
        ["PHASE_CLEAN", "PTEDUCAT"]
    )
)

display(pteducat_out_of_range_by_phase)

# 14. Finalise education cleaning

The only education values outside the documented 0 to 20 year range are ADNI1 special codes `-4` and `-1`.

treat both values as missing and retain the range check as a safeguard for any other invalid entries.

In [ ]:
# Recreate the cleaned education field with explicit handling
# of the observed ADNI special missing-value codes.
ptdemog_working["PTEDUCAT_CLEAN"] = (
    pd.to_numeric(
        ptdemog_working["PTEDUCAT"],
        errors="coerce",
    )
    .replace(
        {
            -4: pd.NA,
            -1: pd.NA,
        }
    )
)

# Retain a range-based QC flag for any unexpected values.
ptdemog_working["PTEDUCAT_OUT_OF_RANGE"] = (
    ptdemog_working["PTEDUCAT_CLEAN"].notna()
    & ~ptdemog_working["PTEDUCAT_CLEAN"].between(0, 20)
)

ptdemog_working.loc[
    ptdemog_working["PTEDUCAT_OUT_OF_RANGE"],
    "PTEDUCAT_CLEAN",
] = pd.NA

print(
    "Remaining education values outside the documented range:",
    int(ptdemog_working["PTEDUCAT_OUT_OF_RANGE"].sum()),
)

print(
    "Missing cleaned education values:",
    int(ptdemog_working["PTEDUCAT_CLEAN"].isna().sum()),
)

# 15. Clean ethnicity and race variables

create cleaned and human-readable versions of the broad ethnicity and race variables.

`PTETHCAT` uses the same coding across ADNI phases. `PTRACCAT` requires phase-aware decoding because ADNI4 separates Native Hawaiian and Other Pacific Islander categories and may store multiple selections separated by `|`.

preserve the original race value and create a decoded label without forcing multi-race responses into a single category.

In [ ]:
# Clean the broad ethnicity category.
ptdemog_working["PTETHCAT_CLEAN"] = (
    pd.to_numeric(
        ptdemog_working["PTETHCAT"],
        errors="coerce",
    )
    .replace(-4, pd.NA)
    .astype("Int64")
)

ptethcat_map = {
    1: "Hispanic or Latino",
    2: "Not Hispanic or Latino",
    3: "Unknown",
}

ptdemog_working["PTETHCAT_LABEL"] = (
    ptdemog_working["PTETHCAT_CLEAN"]
    .map(ptethcat_map)
    .astype("string")
)

# Preserve PTRACCAT as text because some ADNI4 rows contain multiple values,
# for example "1|5".
ptdemog_working["PTRACCAT_CLEAN"] = (
    ptdemog_working["PTRACCAT"]
    .astype("string")
    .str.strip()
    .replace(
        {
            "-4": pd.NA,
            "-4.0": pd.NA,
            "nan": pd.NA,
            "": pd.NA,
        }
    )
)

# Coding used in ADNI1, ADNIGO, ADNI2, and ADNI3.
race_map_legacy = {
    "1": "American Indian or Alaskan Native",
    "2": "Asian",
    "3": "Native Hawaiian or Other Pacific Islander",
    "4": "Black or African American",
    "5": "White",
    "6": "More than one race",
    "7": "Unknown",
}

# Coding used in ADNI4.
race_map_adni4 = {
    "1": "American Indian or Alaskan Native",
    "2": "Asian",
    "4": "Black or African American",
    "5": "White",
    "7": "Unknown",
    "8": "Native Hawaiian",
    "9": "Other Pacific Islander",
}


def decode_race_value(raw_value, phase):
    """
    Decode a single- or multi-select PTRACCAT value using the
    phase-appropriate ADNI dictionary.
    """
    if pd.isna(raw_value):
        return pd.NA

    race_map = (
        race_map_adni4
        if phase == "ADNI4"
        else race_map_legacy
    )

    codes = [
        code.strip()
        for code in str(raw_value).split("|")
        if code.strip()
    ]

    decoded_labels = [
        race_map.get(code, f"Unmapped code {code}")
        for code in codes
    ]

    return " | ".join(decoded_labels)


ptdemog_working["PTRACCAT_LABEL"] = ptdemog_working.apply(
    lambda row: decode_race_value(
        row["PTRACCAT_CLEAN"],
        row["PHASE_CLEAN"],
    ),
    axis=1,
).astype("string")

# Flag rows containing more than one selected race category.
ptdemog_working["PTRACCAT_MULTISELECT"] = (
    ptdemog_working["PTRACCAT_CLEAN"]
    .astype("string")
    .str.contains(r"\|", regex=True, na=False)
)

print(
    "Missing cleaned ethnicity values:",
    int(ptdemog_working["PTETHCAT_CLEAN"].isna().sum()),
)

print(
    "Missing cleaned race values:",
    int(ptdemog_working["PTRACCAT_CLEAN"].isna().sum()),
)

print(
    "Rows with multiple race selections:",
    int(ptdemog_working["PTRACCAT_MULTISELECT"].sum()),
)

display(
    ptdemog_working[
        [
            "PHASE_CLEAN",
            "PTETHCAT",
            "PTETHCAT_CLEAN",
            "PTETHCAT_LABEL",
            "PTRACCAT",
            "PTRACCAT_CLEAN",
            "PTRACCAT_LABEL",
            "PTRACCAT_MULTISELECT",
        ]
    ].head(15)
)

# 16. Check for unmapped race codes

verify that every non-missing `PTRACCAT` code was successfully interpreted using the appropriate ADNI phase dictionary.

Any unmapped value will be reviewed before the cleaned race field is finalised.

In [ ]:
# Identify any decoded race values containing an unmapped code.
unmapped_race_rows = (
    ptdemog_working[
        ptdemog_working["PTRACCAT_LABEL"]
        .astype("string")
        .str.contains(
            "Unmapped code",
            na=False,
        )
    ][
        [
            "RID_CLEAN",
            "PHASE_CLEAN",
            "PTRACCAT",
            "PTRACCAT_CLEAN",
            "PTRACCAT_LABEL",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        by=[
            "PHASE_CLEAN",
            "PTRACCAT_CLEAN",
        ]
    )
)

print(
    "Rows containing an unmapped race code:",
    len(unmapped_race_rows),
)

display(unmapped_race_rows)

# 17. Resolve the legacy race code found in an ADNI4 record

One ADNI4 record contains `PTRACCAT = 3|4|5`.

Code `3` was used in earlier ADNI phases for “Native Hawaiian or Other Pacific Islander,” while ADNI4 normally separates this into codes `8` and `9`. preserve the observed response by decoding code `3` using its legacy meaning and create a flag showing that a legacy race code appeared in an ADNI4 record.

In [ ]:
# Extend the ADNI4 race mapping with the observed legacy code.
race_map_adni4_with_legacy_fallback = {
    **race_map_adni4,
    "3": "Native Hawaiian or Other Pacific Islander",
}


def decode_race_value(raw_value, phase):
    """
    Decode single- or multi-select PTRACCAT values using the
    phase-appropriate ADNI dictionary.

    ADNI4 code 3 is handled as a legacy fallback because one
    observed ADNI4 record retains the earlier combined category.
    """
    if pd.isna(raw_value):
        return pd.NA

    if phase == "ADNI4":
        race_map = race_map_adni4_with_legacy_fallback
    else:
        race_map = race_map_legacy

    codes = [
        code.strip()
        for code in str(raw_value).split("|")
        if code.strip()
    ]

    decoded_labels = [
        race_map.get(code, f"Unmapped code {code}")
        for code in codes
    ]

    return " | ".join(decoded_labels)


# Flag ADNI4 rows containing the legacy combined race code.
ptdemog_working["PTRACCAT_ADNI4_LEGACY_CODE3"] = (
    ptdemog_working["PHASE_CLEAN"].eq("ADNI4")
    & ptdemog_working["PTRACCAT_CLEAN"]
    .astype("string")
    .str.split("|")
    .apply(
        lambda values: "3" in values
        if isinstance(values, list)
        else False
    )
)

# Recreate the decoded race labels.
ptdemog_working["PTRACCAT_LABEL"] = (
    ptdemog_working.apply(
        lambda row: decode_race_value(
            row["PTRACCAT_CLEAN"],
            row["PHASE_CLEAN"],
        ),
        axis=1,
    )
    .astype("string")
)

print(
    "ADNI4 rows containing legacy race code 3:",
    int(
        ptdemog_working[
            "PTRACCAT_ADNI4_LEGACY_CODE3"
        ].sum()
    ),
)

display(
    ptdemog_working.loc[
        ptdemog_working[
            "PTRACCAT_ADNI4_LEGACY_CODE3"
        ],
        [
            "RID_CLEAN",
            "PHASE_CLEAN",
            "PTRACCAT",
            "PTRACCAT_LABEL",
            "PTRACCAT_ADNI4_LEGACY_CODE3",
        ],
    ]
)

# 18. Clean detailed ADNI4 race and ethnicity fields

clean the ADNI4-specific detailed ethnicity and race variables.

These variables are conditional follow-up questions, so they are expected to be missing for most participants. Some responses contain multiple selected categories separated by `|`, so preserve them as multi-label strings and decode each selected value separately.

Only category meanings explicitly confirmed in the ADNI data dictionary will be used.

In [ ]:
# Preserve the detailed ADNI4 variables as cleaned strings.
detailed_multiselect_columns = [
    "PTETHCATH",
    "PTASIAN",
    "PTOPI",
]

for column in detailed_multiselect_columns:
    ptdemog_working[f"{column}_CLEAN"] = (
        ptdemog_working[column]
        .astype("string")
        .str.strip()
        .replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "<NA>": pd.NA,
                "-4": pd.NA,
                "-4.0": pd.NA,
            }
        )
    )

# Define only dictionary-confirmed mappings.
ptethcath_map = {
    "2": "Puerto Rican",
    "3": "Cuban",
    "5": "Unknown Hispanic or Latino origin",
    "6": "Dominican",
    "7": "Mexican",
    "8": "South American",
    "9": "Central American",
    "10": "Other Latin culture or origin",
}

ptasian_map = {
    "1": "Chinese",
    "2": "Filipino",
    "3": "Asian Indian",
    "4": "Vietnamese",
    "5": "Korean",
    "6": "Japanese",
    "7": "Other Asian",
}

ptopi_map = {
    "1": "Samoan",
    "2": "Chamorro",
    "3": "Other Pacific Islander",
}


def decode_multiselect(raw_value, value_map):
    """
    Decode a single- or multi-select categorical value while
    preserving every selected category.
    """
    if pd.isna(raw_value):
        return pd.NA

    codes = [
        code.strip()
        for code in str(raw_value).split("|")
        if code.strip()
    ]

    labels = [
        value_map.get(code, f"Unmapped code {code}")
        for code in codes
    ]

    return " | ".join(labels)


# Decode the detailed ethnicity and race variables.
ptdemog_working["PTETHCATH_LABEL"] = (
    ptdemog_working["PTETHCATH_CLEAN"]
    .apply(
        lambda value: decode_multiselect(
            value,
            ptethcath_map,
        )
    )
    .astype("string")
)

ptdemog_working["PTASIAN_LABEL"] = (
    ptdemog_working["PTASIAN_CLEAN"]
    .apply(
        lambda value: decode_multiselect(
            value,
            ptasian_map,
        )
    )
    .astype("string")
)

ptdemog_working["PTOPI_LABEL"] = (
    ptdemog_working["PTOPI_CLEAN"]
    .apply(
        lambda value: decode_multiselect(
            value,
            ptopi_map,
        )
    )
    .astype("string")
)

# Flag responses containing more than one selected category.
for column in detailed_multiselect_columns:
    ptdemog_working[f"{column}_MULTISELECT"] = (
        ptdemog_working[f"{column}_CLEAN"]
        .astype("string")
        .str.contains(
            r"\|",
            regex=True,
            na=False,
        )
    )

# Summarise coverage and the number of multi-select responses.
detailed_demographic_summary = pd.DataFrame(
    {
        "VARIABLE": detailed_multiselect_columns,
        "NON_MISSING_ROWS": [
            int(
                ptdemog_working[
                    f"{column}_CLEAN"
                ].notna().sum()
            )
            for column in detailed_multiselect_columns
        ],
        "MULTISELECT_ROWS": [
            int(
                ptdemog_working[
                    f"{column}_MULTISELECT"
                ].sum()
            )
            for column in detailed_multiselect_columns
        ],
    }
)

display(detailed_demographic_summary)

# Display non-missing detailed demographic responses.
display(
    ptdemog_working[
        [
            "RID_CLEAN",
            "PHASE_CLEAN",
            "PTETHCATH_CLEAN",
            "PTETHCATH_LABEL",
            "PTETHCATH_MULTISELECT",
            "PTASIAN_CLEAN",
            "PTASIAN_LABEL",
            "PTASIAN_MULTISELECT",
            "PTOPI_CLEAN",
            "PTOPI_LABEL",
            "PTOPI_MULTISELECT",
        ]
    ]
    .dropna(
        subset=[
            "PTETHCATH_CLEAN",
            "PTASIAN_CLEAN",
            "PTOPI_CLEAN",
        ],
        how="all",
    )
    .reset_index(drop=True)
)


# 19. Check detailed demographic fields for unmapped codes

verify that every non-missing detailed ethnicity and race code was successfully decoded.

Any unmapped value will be reviewed rather than silently discarded or assigned an assumed meaning.

In [ ]:
# Identify rows containing any unmapped detailed demographic code.
unmapped_detailed_rows = (
    ptdemog_working[
        ptdemog_working[
            [
                "PTETHCATH_LABEL",
                "PTASIAN_LABEL",
                "PTOPI_LABEL",
            ]
        ]
        .astype("string")
        .apply(
            lambda column: column.str.contains(
                "Unmapped code",
                na=False,
            )
        )
        .any(axis=1)
    ]
    [
        [
            "RID_CLEAN",
            "PHASE_CLEAN",
            "PTETHCATH_CLEAN",
            "PTETHCATH_LABEL",
            "PTASIAN_CLEAN",
            "PTASIAN_LABEL",
            "PTOPI_CLEAN",
            "PTOPI_LABEL",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Rows with unmapped detailed demographic codes:",
    len(unmapped_detailed_rows),
)

display(unmapped_detailed_rows)

# 20. Correct detailed race and ethnicity decoding

The detailed ADNI4 fields contain two coding issues.

`PTETHCATH` includes observed values `1` and `4`, although their meanings are not provided in the extracted data-dictionary definition. preserve these values as unresolved documented codes rather than assigning assumed meanings.

`PTOPI` is stored numerically, producing values such as `3.0`. normalise integer-like codes so that `3.0` is interpreted as code `3`.

In [ ]:
# Detailed ADNI4 variables that may contain single or multiple selections.
detailed_multiselect_columns = [
    "PTETHCATH",
    "PTASIAN",
    "PTOPI",
]


def normalise_categorical_code(value):
    """
    Convert categorical values to consistent strings.

    Examples:
        3.0   -> "3"
        "3.0" -> "3"
        "3|7" -> "3|7"
    """
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    if text in {"", "nan", "<NA>", "-4", "-4.0"}:
        return pd.NA

    normalised_codes = []

    for code in text.split("|"):
        code = code.strip()

        try:
            numeric_code = float(code)

            if numeric_code.is_integer():
                code = str(int(numeric_code))
        except ValueError:
            pass

        normalised_codes.append(code)

    return "|".join(normalised_codes)


# Create consistently formatted cleaned values.
for column in detailed_multiselect_columns:
    ptdemog_working[f"{column}_CLEAN"] = (
        ptdemog_working[column]
        .apply(normalise_categorical_code)
        .astype("string")
    )

# Use only meanings explicitly confirmed by the extracted dictionary.
ptethcath_map = {
    "1": "Unresolved Hispanic or Latino origin code 1",
    "2": "Puerto Rican",
    "3": "Cuban",
    "4": "Unresolved Hispanic or Latino origin code 4",
    "5": "Unknown Hispanic or Latino origin",
    "6": "Dominican",
    "7": "Mexican",
    "8": "South American",
    "9": "Central American",
    "10": "Other Latin culture or origin",
}

ptasian_map = {
    "1": "Chinese",
    "2": "Filipino",
    "3": "Asian Indian",
    "4": "Vietnamese",
    "5": "Korean",
    "6": "Japanese",
    "7": "Other Asian",
}

ptopi_map = {
    "1": "Samoan",
    "2": "Chamorro",
    "3": "Other Pacific Islander",
}


def decode_multiselect(raw_value, value_map):
    """
    Decode every category in a single- or multi-select response.
    """
    if pd.isna(raw_value):
        return pd.NA

    codes = [
        code.strip()
        for code in str(raw_value).split("|")
        if code.strip()
    ]

    labels = [
        value_map.get(code, f"Unmapped code {code}")
        for code in codes
    ]

    return " | ".join(labels)


# Decode the cleaned values.
ptdemog_working["PTETHCATH_LABEL"] = (
    ptdemog_working["PTETHCATH_CLEAN"]
    .apply(lambda value: decode_multiselect(value, ptethcath_map))
    .astype("string")
)

ptdemog_working["PTASIAN_LABEL"] = (
    ptdemog_working["PTASIAN_CLEAN"]
    .apply(lambda value: decode_multiselect(value, ptasian_map))
    .astype("string")
)

ptdemog_working["PTOPI_LABEL"] = (
    ptdemog_working["PTOPI_CLEAN"]
    .apply(lambda value: decode_multiselect(value, ptopi_map))
    .astype("string")
)

# Flag multi-select values.
for column in detailed_multiselect_columns:
    ptdemog_working[f"{column}_MULTISELECT"] = (
        ptdemog_working[f"{column}_CLEAN"]
        .str.contains(r"\|", regex=True, na=False)
    )

# Explicitly flag the two PTETHCATH codes whose meanings
# were absent from the extracted dictionary definition.
ptdemog_working["PTETHCATH_UNRESOLVED_CODE"] = (
    ptdemog_working["PTETHCATH_CLEAN"]
    .fillna("")
    .str.split("|")
    .apply(
        lambda codes: any(
            code in {"1", "4"}
            for code in codes
        )
    )
)

# Summarise coverage.
detailed_demographic_summary = pd.DataFrame(
    {
        "VARIABLE": detailed_multiselect_columns,
        "NON_MISSING_ROWS": [
            int(ptdemog_working[f"{column}_CLEAN"].notna().sum())
            for column in detailed_multiselect_columns
        ],
        "MULTISELECT_ROWS": [
            int(ptdemog_working[f"{column}_MULTISELECT"].sum())
            for column in detailed_multiselect_columns
        ],
    }
)

display(detailed_demographic_summary)

print(
    "Rows containing unresolved PTETHCATH codes 1 or 4:",
    int(ptdemog_working["PTETHCATH_UNRESOLVED_CODE"].sum()),
)

# 21. Clean disease-history fields for audit only

clean the PTDEMOG disease-history variables while keeping them separate from the model-facing demographic features.

`PTCOGBEG` records the estimated year of cognitive-symptom onset, `PTADBEG` records the estimated year of Alzheimer's disease symptom onset, and `PTADDX` records the year of Alzheimer's disease diagnosis.

The special value `9999` has a documented meaning rather than being an ordinary year. preserve that meaning through separate status flags and convert it to missing only in the cleaned numeric year fields.

These variables will remain excluded from the predictor set because they directly describe cognitive or Alzheimer's disease history.

In [ ]:
# Convert the disease-history variables to numeric form.
for column in ["PTCOGBEG", "PTADBEG", "PTADDX"]:
    ptdemog_working[f"{column}_NUMERIC"] = pd.to_numeric(
        ptdemog_working[column],
        errors="coerce",
    )

# Preserve the documented semantic meaning of 9999.
ptdemog_working["PTCOGBEG_COGNITIVELY_NORMAL"] = (
    ptdemog_working["PTCOGBEG_NUMERIC"].eq(9999)
)

ptdemog_working["PTADDX_NOT_DIAGNOSED_WITH_AD"] = (
    ptdemog_working["PTADDX_NUMERIC"].eq(9999)
)

# Create cleaned year fields.
ptdemog_working["PTCOGBEG_YEAR_CLEAN"] = (
    ptdemog_working["PTCOGBEG_NUMERIC"]
    .replace(
        {
            9999: pd.NA,
            -4: pd.NA,
            -1: pd.NA,
        }
    )
    .astype("Int64")
)

ptdemog_working["PTADBEG_YEAR_CLEAN"] = (
    ptdemog_working["PTADBEG_NUMERIC"]
    .replace(
        {
            9999: pd.NA,
            -4: pd.NA,
            -1: pd.NA,
        }
    )
    .astype("Int64")
)

ptdemog_working["PTADDX_YEAR_CLEAN"] = (
    ptdemog_working["PTADDX_NUMERIC"]
    .replace(
        {
            9999: pd.NA,
            -4: pd.NA,
            -1: pd.NA,
        }
    )
    .astype("Int64")
)

# Flag implausible calendar years without deleting the raw values.
year_columns = [
    "PTCOGBEG_YEAR_CLEAN",
    "PTADBEG_YEAR_CLEAN",
    "PTADDX_YEAR_CLEAN",
]

for column in year_columns:
    ptdemog_working[f"{column}_OUT_OF_RANGE"] = (
        ptdemog_working[column].notna()
        & ~ptdemog_working[column].between(1900, 2026)
    )

    ptdemog_working.loc[
        ptdemog_working[f"{column}_OUT_OF_RANGE"],
        column,
    ] = pd.NA

# Summarise the resulting fields.
disease_history_summary = pd.DataFrame(
    {
        "VARIABLE": [
            "PTCOGBEG",
            "PTADBEG",
            "PTADDX",
        ],
        "CLEAN_YEAR_NON_MISSING": [
            int(ptdemog_working["PTCOGBEG_YEAR_CLEAN"].notna().sum()),
            int(ptdemog_working["PTADBEG_YEAR_CLEAN"].notna().sum()),
            int(ptdemog_working["PTADDX_YEAR_CLEAN"].notna().sum()),
        ],
        "SPECIAL_STATUS_COUNT": [
            int(
                ptdemog_working[
                    "PTCOGBEG_COGNITIVELY_NORMAL"
                ].sum()
            ),
            0,
            int(
                ptdemog_working[
                    "PTADDX_NOT_DIAGNOSED_WITH_AD"
                ].sum()
            ),
        ],
        "OUT_OF_RANGE_COUNT": [
            int(
                ptdemog_working[
                    "PTCOGBEG_YEAR_CLEAN_OUT_OF_RANGE"
                ].sum()
            ),
            int(
                ptdemog_working[
                    "PTADBEG_YEAR_CLEAN_OUT_OF_RANGE"
                ].sum()
            ),
            int(
                ptdemog_working[
                    "PTADDX_YEAR_CLEAN_OUT_OF_RANGE"
                ].sum()
            ),
        ],
    }
)

display(disease_history_summary)

# 22. Clean phase-specific language and employment variables

clean the remaining PTDEMOG variables that may be useful for descriptive analysis, quality control, or later ablation experiments.

These variables have limited or phase-specific coverage, so they will not be included in the initial predictor set. The original raw values will remain unchanged.

clean the ADNI4 monolingual-English indicator, the age at which English was learned, the coded occupation category, and the retirement year.

In [ ]:
# Clean the ADNI4 monolingual-English indicator.
ptdemog_working["PTENGSPK_CLEAN"] = (
    pd.to_numeric(
        ptdemog_working["PTENGSPK"],
        errors="coerce",
    )
    .replace(
        {
            -4: pd.NA,
            -1: pd.NA,
        }
    )
    .astype("Int64")
)

ptengspk_map = {
    0: "No",
    1: "Yes",
}

ptdemog_working["PTENGSPK_LABEL"] = (
    ptdemog_working["PTENGSPK_CLEAN"]
    .map(ptengspk_map)
    .astype("string")
)

# Clean the age at which English was learned.
ptdemog_working["PTENGSPKAGE_CLEAN"] = (
    pd.to_numeric(
        ptdemog_working["PTENGSPKAGE"],
        errors="coerce",
    )
    .replace(
        {
            -4: pd.NA,
            -1: pd.NA,
        }
    )
)

ptdemog_working["PTENGSPKAGE_OUT_OF_RANGE"] = (
    ptdemog_working["PTENGSPKAGE_CLEAN"].notna()
    & ~ptdemog_working["PTENGSPKAGE_CLEAN"].between(0, 90)
)

ptdemog_working.loc[
    ptdemog_working["PTENGSPKAGE_OUT_OF_RANGE"],
    "PTENGSPKAGE_CLEAN",
] = pd.NA

# Clean the ADNI4 coded occupation category.
# Code 9 means missing or unknown and is therefore converted to missing.
ptdemog_working["PTWORK_CLEAN"] = (
    pd.to_numeric(
        ptdemog_working["PTWORK"],
        errors="coerce",
    )
    .replace(
        {
            -4: pd.NA,
            -1: pd.NA,
            9: pd.NA,
        }
    )
    .astype("Int64")
)

ptwork_map = {
    1: "Professional or higher executive",
    2: "Middle professional or small business owner",
    3: "Manager",
    4: "Support personnel, drafter, or technician",
    5: "Non-professional arts, design, entertainment, or sports",
    6: "Aide, assistant, or clerk",
    7: "Labourer",
    8: "Other or never employed",
}

ptdemog_working["PTWORK_LABEL"] = (
    ptdemog_working["PTWORK_CLEAN"]
    .map(ptwork_map)
    .astype("string")
)

# Parse the retirement field and extract the retirement year.
ptdemog_working["PTRTYR_PARSED"] = pd.to_datetime(
    ptdemog_working["PTRTYR"],
    errors="coerce",
)

ptdemog_working["PTRTYR_YEAR_CLEAN"] = (
    ptdemog_working["PTRTYR_PARSED"]
    .dt.year
    .astype("Int64")
)

# Flag retirement years outside a plausible calendar-year range.
ptdemog_working["PTRTYR_YEAR_OUT_OF_RANGE"] = (
    ptdemog_working["PTRTYR_YEAR_CLEAN"].notna()
    & ~ptdemog_working["PTRTYR_YEAR_CLEAN"].between(1900, 2026)
)

ptdemog_working.loc[
    ptdemog_working["PTRTYR_YEAR_OUT_OF_RANGE"],
    "PTRTYR_YEAR_CLEAN",
] = pd.NA

# Summarise coverage and QC results.
phase_specific_summary = pd.DataFrame(
    {
        "VARIABLE": [
            "PTENGSPK",
            "PTENGSPKAGE",
            "PTWORK",
            "PTRTYR",
        ],
        "CLEAN_NON_MISSING_ROWS": [
            int(ptdemog_working["PTENGSPK_CLEAN"].notna().sum()),
            int(ptdemog_working["PTENGSPKAGE_CLEAN"].notna().sum()),
            int(ptdemog_working["PTWORK_CLEAN"].notna().sum()),
            int(ptdemog_working["PTRTYR_YEAR_CLEAN"].notna().sum()),
        ],
        "OUT_OF_RANGE_ROWS": [
            0,
            int(ptdemog_working["PTENGSPKAGE_OUT_OF_RANGE"].sum()),
            0,
            int(ptdemog_working["PTRTYR_YEAR_OUT_OF_RANGE"].sum()),
        ],
    }
)

display(phase_specific_summary)

# 23. Create the cleaned handedness variable

The participant-consistency check requires a cleaned handedness field. create it from the raw `PTHAND` values using the dictionary-confirmed coding and convert the special values `-4` and `-1` to missing.

In [ ]:
ptdemog_working["PTHAND_CLEAN"] = (
    pd.to_numeric(
        ptdemog_working["PTHAND"],
        errors="coerce",
    )
    .replace(
        {
            -4: pd.NA,
            -1: pd.NA,
        }
    )
    .astype("Int64")
)

pthand_map = {
    1: "Right",
    2: "Left",
}

ptdemog_working["PTHAND_LABEL"] = (
    ptdemog_working["PTHAND_CLEAN"]
    .map(pthand_map)
    .astype("string")
)

display(
    ptdemog_working[
        [
            "PTHAND",
            "PTHAND_CLEAN",
            "PTHAND_LABEL",
        ]
    ].head()
)

# 24. Create missing readable labels for conflict review

The conflict-review table uses readable labels for sex and primary language. create those label columns from the already cleaned numeric fields before displaying the conflicting records.

In [ ]:
# Create readable labels for sex.
ptgender_map = {
    1: "Male",
    2: "Female",
}

ptdemog_working["PTGENDER_LABEL"] = (
    ptdemog_working["PTGENDER_CLEAN"]
    .map(ptgender_map)
    .astype("string")
)

# Create readable labels for primary language.
ptplang_map = {
    1: "English",
    2: "Spanish",
    3: "Other",
}

ptdemog_working["PTPLANG_LABEL"] = (
    ptdemog_working["PTPLANG_CLEAN"]
    .map(ptplang_map)
    .astype("string")
)

print("Missing label columns created.")

# 25. Check participant-level consistency in stable demographic variables

check whether demographic variables that should usually remain stable agree across each participant's repeated PTDEMOG records.

The check will cover birth year, sex, handedness, primary language, ethnicity, and race. A participant will be flagged when more than one distinct non-missing value appears across their records.

These flags will be used for quality control only. Records will not be removed or automatically overwritten.

In [ ]:
# Create one birth-year field from the two matching source fields.
# PTDOB_YEAR is preferred, with PTDOBYY_YEAR used when PTDOB_YEAR is missing.
ptdemog_working["BIRTH_YEAR_CLEAN"] = (
    ptdemog_working["PTDOB_YEAR"]
    .combine_first(ptdemog_working["PTDOBYY_YEAR"])
    .astype("Int64")
)

# Variables expected to remain stable for a participant.
stable_demographic_columns = {
    "BIRTH_YEAR_CLEAN": "Birth year",
    "PTGENDER_CLEAN": "Sex",
    "PTHAND_CLEAN": "Handedness",
    "PTPLANG_CLEAN": "Primary language",
    "PTETHCAT_CLEAN": "Ethnicity",
    "PTRACCAT_CLEAN": "Race",
}

# Count the number of distinct non-missing values recorded per participant.
participant_consistency = (
    ptdemog_working
    .groupby("RID_CLEAN")
    .agg(
        PTID=("PTID_CLEAN", "first"),
        PTDEMOG_ROW_COUNT=("RID_CLEAN", "size"),
        **{
            f"{column}_N_UNIQUE": (
                column,
                lambda values: values.dropna().nunique()
            )
            for column in stable_demographic_columns
        },
    )
    .reset_index()
)

# Create one conflict flag per variable.
for column in stable_demographic_columns:
    participant_consistency[f"{column}_CONFLICT"] = (
        participant_consistency[f"{column}_N_UNIQUE"] > 1
    )

# Create an overall participant-level conflict flag.
conflict_columns = [
    f"{column}_CONFLICT"
    for column in stable_demographic_columns
]

participant_consistency["ANY_STABLE_DEMOGRAPHIC_CONFLICT"] = (
    participant_consistency[conflict_columns].any(axis=1)
)

# Summarise conflicts by variable.
consistency_summary = pd.DataFrame(
    {
        "VARIABLE": list(stable_demographic_columns.keys()),
        "DESCRIPTION": list(stable_demographic_columns.values()),
        "PARTICIPANTS_WITH_CONFLICT": [
            int(
                participant_consistency[
                    f"{column}_CONFLICT"
                ].sum()
            )
            for column in stable_demographic_columns
        ],
    }
)

print(
    "Participants with at least one stable demographic conflict:",
    int(
        participant_consistency[
            "ANY_STABLE_DEMOGRAPHIC_CONFLICT"
        ].sum()
    ),
)

display(consistency_summary)

# 26. Inspect participant-level demographic conflicts

inspect the repeated PTDEMOG records for participants whose stable demographic values differ across visits.

begin with birth year, sex, and handedness because these fields should generally remain constant. Language, ethnicity, and race conflicts will also be displayed, but they will not automatically be treated as data errors because reporting practices changed across ADNI phases.

In [ ]:
# Define the fields required for conflict review.
conflict_review_columns = [
    "RID_CLEAN",
    "PTID_CLEAN",
    "PHASE_CLEAN",
    "VISCODE2_CLEAN",
    "VISDATE_PARSED",
    "BIRTH_YEAR_CLEAN",
    "PTGENDER_CLEAN",
    "PTGENDER_LABEL",
    "PTHAND_CLEAN",
    "PTHAND_LABEL",
    "PTPLANG_CLEAN",
    "PTPLANG_LABEL",
    "PTETHCAT_CLEAN",
    "PTETHCAT_LABEL",
    "PTRACCAT_CLEAN",
    "PTRACCAT_LABEL",
]

# Identify participants with any stable demographic conflict.
conflicting_rids = participant_consistency.loc[
    participant_consistency["ANY_STABLE_DEMOGRAPHIC_CONFLICT"],
    "RID_CLEAN",
]

# Retrieve all relevant PTDEMOG records for those participants.
ptdemog_conflict_records = (
    ptdemog_working.loc[
        ptdemog_working["RID_CLEAN"].isin(conflicting_rids),
        conflict_review_columns,
    ]
    .sort_values(
        by=[
            "RID_CLEAN",
            "VISDATE_PARSED",
            "PHASE_CLEAN",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

# Identify participants with birth-year, sex, or handedness conflicts.
high_priority_rids = participant_consistency.loc[
    participant_consistency[
        [
            "BIRTH_YEAR_CLEAN_CONFLICT",
            "PTGENDER_CLEAN_CONFLICT",
            "PTHAND_CLEAN_CONFLICT",
        ]
    ].any(axis=1),
    "RID_CLEAN",
]

high_priority_conflicts = (
    ptdemog_conflict_records[
        ptdemog_conflict_records["RID_CLEAN"].isin(high_priority_rids)
    ]
    .reset_index(drop=True)
)

print(
    "Participants with birth-year, sex, or handedness conflicts:",
    high_priority_conflicts["RID_CLEAN"].nunique(),
)

display(high_priority_conflicts)

In [ ]:
# Identify participants with primary-language, ethnicity, or race conflicts.
lower_priority_rids = participant_consistency.loc[
    participant_consistency[
        [
            "PTPLANG_CLEAN_CONFLICT",
            "PTETHCAT_CLEAN_CONFLICT",
            "PTRACCAT_CLEAN_CONFLICT",
        ]
    ].any(axis=1),
    "RID_CLEAN",
]

language_ethnicity_race_conflicts = (
    ptdemog_conflict_records[
        ptdemog_conflict_records["RID_CLEAN"].isin(lower_priority_rids)
    ]
    .reset_index(drop=True)
)

print(
    "Participants with primary-language, ethnicity, or race conflicts:",
    language_ethnicity_race_conflicts["RID_CLEAN"].nunique(),
)

display(language_ethnicity_race_conflicts)

# 27. Summary of participant-level demographic inconsistencies

I checked whether demographic variables that should usually remain stable were recorded consistently across repeated PTDEMOG rows for the same participant.

In total, 54 participants had at least one conflict across birth year, sex, handedness, primary language, ethnicity, or race.

The most important inconsistencies were:

- 4 participants with conflicting birth years
- 1 participant with conflicting sex values
- 8 participants with conflicting handedness values
- 13 participants with conflicting primary-language values
- 8 participants with conflicting ethnicity values
- 25 participants with conflicting race values

The birth-year conflicts ranged from one-year differences to larger four-year differences. These values should not be resolved automatically because they may reflect data-entry corrections, phase transitions, or differences in how demographic information was recorded.

One participant had sex recorded as female in ADNI2 and male in ADNI4. This is a high-priority inconsistency that should be checked against another trusted source before a final participant-level value is selected.

Handedness conflicts were also observed, with several participants changing between right- and left-handed across phases. Because handedness is not central to the initial model and appears inconsistently recorded, it should remain an optional variable and should not be used without later resolution.

Primary-language, ethnicity, and race conflicts require more cautious interpretation. Some differences may reflect genuine corrections, but others are likely caused by changes in data collection between ADNI phases. In particular, ADNI4 allows more detailed and multi-select race responses, so a participant previously recorded as “More than one race” may later appear as a specific combination such as “Asian | White.” These cases should not automatically be treated as errors.

No conflicting value will be overwritten at this stage. All longitudinal records will be preserved, and participant-level conflict flags will be added for later quality control and reference-date selection. Birth-year and sex conflicts will be treated as high priority, while handedness, language, ethnicity, and race conflicts will remain available for later review.

# 28. Add participant-level consistency flags to the longitudinal table

attach the participant-level consistency results to every PTDEMOG row.

These flags will preserve information about conflicts without changing or overwriting any recorded demographic value. Birth-year and sex conflicts will be treated as high-priority quality-control issues, while handedness, language, ethnicity, and race conflicts will remain available for later review.

In [ ]:
# Select the participant-level consistency flags to merge back.
participant_conflict_flags = participant_consistency[
    [
        "RID_CLEAN",
        "BIRTH_YEAR_CLEAN_CONFLICT",
        "PTGENDER_CLEAN_CONFLICT",
        "PTHAND_CLEAN_CONFLICT",
        "PTPLANG_CLEAN_CONFLICT",
        "PTETHCAT_CLEAN_CONFLICT",
        "PTRACCAT_CLEAN_CONFLICT",
        "ANY_STABLE_DEMOGRAPHIC_CONFLICT",
    ]
].copy()

# Define a high-priority conflict flag for variables expected
# to be biologically or administratively stable.
participant_conflict_flags["HIGH_PRIORITY_DEMOGRAPHIC_CONFLICT"] = (
    participant_conflict_flags[
        [
            "BIRTH_YEAR_CLEAN_CONFLICT",
            "PTGENDER_CLEAN_CONFLICT",
        ]
    ].any(axis=1)
)

# Merge the participant-level flags back into every longitudinal row.
ptdemog_working = ptdemog_working.merge(
    participant_conflict_flags,
    on="RID_CLEAN",
    how="left",
    validate="many_to_one",
)

# Replace any theoretically missing flags with False.
conflict_flag_columns = [
    "BIRTH_YEAR_CLEAN_CONFLICT",
    "PTGENDER_CLEAN_CONFLICT",
    "PTHAND_CLEAN_CONFLICT",
    "PTPLANG_CLEAN_CONFLICT",
    "PTETHCAT_CLEAN_CONFLICT",
    "PTRACCAT_CLEAN_CONFLICT",
    "ANY_STABLE_DEMOGRAPHIC_CONFLICT",
    "HIGH_PRIORITY_DEMOGRAPHIC_CONFLICT",
]

ptdemog_working[conflict_flag_columns] = (
    ptdemog_working[conflict_flag_columns]
    .fillna(False)
    .astype(bool)
)

print(
    "Participants with high-priority birth-year or sex conflicts:",
    participant_conflict_flags[
        "HIGH_PRIORITY_DEMOGRAPHIC_CONFLICT"
    ].sum(),
)

print(
    "Participants with any reviewed demographic conflict:",
    participant_conflict_flags[
        "ANY_STABLE_DEMOGRAPHIC_CONFLICT"
    ].sum(),
)

# 29. Confirm the cleaned PTDEMOG fields

confirm that the cleaned variables and quality-control flags created throughout the notebook are present before generating the final outputs.

This is a final completeness check only. It will not modify the data.

In [ ]:
expected_clean_columns = [
    # Identifiers and dates
    "RID_CLEAN",
    "PTID_CLEAN",
    "PHASE_CLEAN",
    "VISCODE_CLEAN",
    "VISCODE2_CLEAN",
    "VISDATE_PARSED",
    "PTDEMOG_DATE_STATUS",

    # Birth year
    "BIRTH_YEAR_CLEAN",
    "BIRTH_YEAR_CONFLICT",

    # Core demographics
    "PTGENDER_CLEAN",
    "PTGENDER_LABEL",
    "PTEDUCAT_CLEAN",
    "PTTLANG_CLEAN",
    "PTTLANG_LABEL",
    "PTPLANG_CLEAN",
    "PTPLANG_LABEL",

    # Optional demographics
    "PTHAND_CLEAN",
    "PTHAND_LABEL",
    "PTMARRY_CLEAN",
    "PTMARRY_LABEL",
    "PTNLANG_CLEAN",
    "PTNLANG_LABEL",
    "PTLANGTTL_CLEAN",
    "PTNOTRT_CLEAN",
    "PTNOTRT_LABEL",

    # Race and ethnicity
    "PTETHCAT_CLEAN",
    "PTETHCAT_LABEL",
    "PTRACCAT_CLEAN",
    "PTRACCAT_LABEL",
    "PTRACCAT_MULTISELECT",
    "PTETHCATH_CLEAN",
    "PTETHCATH_LABEL",
    "PTASIAN_CLEAN",
    "PTASIAN_LABEL",
    "PTOPI_CLEAN",
    "PTOPI_LABEL",

    # Phase-specific fields
    "PTENGSPK_CLEAN",
    "PTENGSPK_LABEL",
    "PTENGSPKAGE_CLEAN",
    "PTWORK_CLEAN",
    "PTWORK_LABEL",
    "PTRTYR_YEAR_CLEAN",

    # Disease-history audit fields
    "PTCOGBEG_YEAR_CLEAN",
    "PTCOGBEG_COGNITIVELY_NORMAL",
    "PTADBEG_YEAR_CLEAN",
    "PTADDX_YEAR_CLEAN",
    "PTADDX_NOT_DIAGNOSED_WITH_AD",

    # Participant-level consistency flags
    "BIRTH_YEAR_CLEAN_CONFLICT",
    "PTGENDER_CLEAN_CONFLICT",
    "PTHAND_CLEAN_CONFLICT",
    "PTPLANG_CLEAN_CONFLICT",
    "PTETHCAT_CLEAN_CONFLICT",
    "PTRACCAT_CLEAN_CONFLICT",
    "ANY_STABLE_DEMOGRAPHIC_CONFLICT",
    "HIGH_PRIORITY_DEMOGRAPHIC_CONFLICT",
]

missing_clean_columns = [
    column
    for column in expected_clean_columns
    if column not in ptdemog_working.columns
]

print(f"Expected cleaned columns: {len(expected_clean_columns)}")
print(f"Missing cleaned columns: {len(missing_clean_columns)}")

if missing_clean_columns:
    print("\nMissing columns:")
    for column in missing_clean_columns:
        print(f"- {column}")
else:
    print("\nAll expected cleaned PTDEMOG columns are present.")

# 30. Create the remaining cleaned optional demographic fields

The final completeness check identified eight cleaned fields that have not yet been created in the current notebook state.

create the testing-language label, marital-status fields, native-language fields, total-language count, and retirement-status fields using the coding confirmed in the ADNI data dictionary. The original raw columns will remain unchanged.

In [ ]:
# Create the readable testing-language label.
pttlang_map = {
    1: "English",
    2: "Spanish",
}

ptdemog_working["PTTLANG_LABEL"] = (
    ptdemog_working["PTTLANG_CLEAN"]
    .map(pttlang_map)
    .astype("string")
)

# Clean and decode marital status.
ptdemog_working["PTMARRY_CLEAN"] = (
    pd.to_numeric(
        ptdemog_working["PTMARRY"],
        errors="coerce",
    )
    .replace(
        {
            -4: pd.NA,
            -1: pd.NA,
        }
    )
    .astype("Int64")
)

ptmarry_map = {
    1: "Married",
    2: "Widowed",
    3: "Divorced",
    4: "Never married",
    5: "Unknown",
    6: "Domestic partnership",
}

ptdemog_working["PTMARRY_LABEL"] = (
    ptdemog_working["PTMARRY_CLEAN"]
    .map(ptmarry_map)
    .astype("string")
)

# Clean and decode native language.
ptdemog_working["PTNLANG_CLEAN"] = (
    pd.to_numeric(
        ptdemog_working["PTNLANG"],
        errors="coerce",
    )
    .replace(
        {
            -4: pd.NA,
            -1: pd.NA,
        }
    )
    .astype("Int64")
)

ptnlang_map = {
    1: "English",
    2: "Spanish",
    3: "Other",
}

ptdemog_working["PTNLANG_LABEL"] = (
    ptdemog_working["PTNLANG_CLEAN"]
    .map(ptnlang_map)
    .astype("string")
)

# Clean the total number of languages spoken.
ptdemog_working["PTLANGTTL_CLEAN"] = (
    pd.to_numeric(
        ptdemog_working["PTLANGTTL"],
        errors="coerce",
    )
    .replace(
        {
            -4: pd.NA,
            -1: pd.NA,
        }
    )
)

ptdemog_working["PTLANGTTL_OUT_OF_RANGE"] = (
    ptdemog_working["PTLANGTTL_CLEAN"].notna()
    & ~ptdemog_working["PTLANGTTL_CLEAN"].between(1, 6)
)

ptdemog_working.loc[
    ptdemog_working["PTLANGTTL_OUT_OF_RANGE"],
    "PTLANGTTL_CLEAN",
] = pd.NA

# Clean and decode retirement status.
ptdemog_working["PTNOTRT_CLEAN"] = (
    pd.to_numeric(
        ptdemog_working["PTNOTRT"],
        errors="coerce",
    )
    .replace(
        {
            -4: pd.NA,
            -1: pd.NA,
        }
    )
    .astype("Int64")
)

ptnotrt_map = {
    0: "No",
    1: "Yes",
    2: "Not applicable",
}

ptdemog_working["PTNOTRT_LABEL"] = (
    ptdemog_working["PTNOTRT_CLEAN"]
    .map(ptnotrt_map)
    .astype("string")
)

print(
    "Language-count values outside the documented range:",
    int(ptdemog_working["PTLANGTTL_OUT_OF_RANGE"].sum()),
)

print("Remaining optional demographic fields created successfully.")

display(
    ptdemog_working[
        [
            "PTTLANG_CLEAN",
            "PTTLANG_LABEL",
            "PTMARRY_CLEAN",
            "PTMARRY_LABEL",
            "PTNLANG_CLEAN",
            "PTNLANG_LABEL",
            "PTLANGTTL_CLEAN",
            "PTNOTRT_CLEAN",
            "PTNOTRT_LABEL",
        ]
    ].head(10)
)

# 31. Check duplicate records and recorded QC errors

check for exact duplicate rows and review records marked with an active quality-control error.

Exact duplicates will be identified using the original PTDEMOG columns rather than notebook-derived fields. No rows will be removed automatically.

The `HAS_QC_ERROR` field is available only for ADNI4 and will be retained as a quality-control flag.

In [ ]:
# Use the original retained PTDEMOG columns for the duplicate check.
duplicate_check_columns = [
    column
    for column in selected_columns
    if column in ptdemog_working.columns
]

ptdemog_working["EXACT_DUPLICATE_RECORD"] = (
    ptdemog_working.duplicated(
        subset=duplicate_check_columns,
        keep=False,
    )
)

exact_duplicate_records = (
    ptdemog_working[
        ptdemog_working["EXACT_DUPLICATE_RECORD"]
    ]
    .sort_values(
        by=[
            "RID_CLEAN",
            "VISDATE_PARSED",
            "PHASE_CLEAN",
        ],
        na_position="last",
    )
    .copy()
)

# Clean the ADNI4 QC flag.
ptdemog_working["HAS_QC_ERROR_CLEAN"] = (
    pd.to_numeric(
        ptdemog_working["HAS_QC_ERROR"],
        errors="coerce",
    )
    .astype("Int64")
)

active_qc_error_records = (
    ptdemog_working[
        ptdemog_working["HAS_QC_ERROR_CLEAN"].eq(1)
    ]
    [
        [
            "RID_CLEAN",
            "PTID_CLEAN",
            "PHASE_CLEAN",
            "VISCODE2_CLEAN",
            "VISDATE_PARSED",
            "HAS_QC_ERROR_CLEAN",
            "DD_CRF_VERSION_LABEL",
        ]
    ]
    .sort_values(
        by=[
            "RID_CLEAN",
            "VISDATE_PARSED",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

print(
    "Rows involved in exact duplicate groups:",
    len(exact_duplicate_records),
)

print(
    "Records with an active QC error:",
    len(active_qc_error_records),
)

display(active_qc_error_records)

# 32. Summarise PTDEMOG missingness and coverage

generate overall and phase-specific missingness summaries for the cleaned demographic variables.

These summaries will show which features have broad cross-phase coverage and which are sparse or limited to later ADNI phases.

In [ ]:
summary_columns = [
    "BIRTH_YEAR_CLEAN",
    "PTGENDER_CLEAN",
    "PTEDUCAT_CLEAN",
    "PTTLANG_CLEAN",
    "PTPLANG_CLEAN",
    "PTHAND_CLEAN",
    "PTMARRY_CLEAN",
    "PTNLANG_CLEAN",
    "PTLANGTTL_CLEAN",
    "PTNOTRT_CLEAN",
    "PTETHCAT_CLEAN",
    "PTRACCAT_CLEAN",
    "PTETHCATH_CLEAN",
    "PTASIAN_CLEAN",
    "PTOPI_CLEAN",
    "PTENGSPK_CLEAN",
    "PTENGSPKAGE_CLEAN",
    "PTWORK_CLEAN",
    "PTRTYR_YEAR_CLEAN",
]

# Overall row-level missingness.
ptdemog_missingness_overall = pd.DataFrame(
    {
        "VARIABLE": summary_columns,
        "NON_MISSING_ROWS": [
            int(ptdemog_working[column].notna().sum())
            for column in summary_columns
        ],
        "MISSING_ROWS": [
            int(ptdemog_working[column].isna().sum())
            for column in summary_columns
        ],
    }
)

ptdemog_missingness_overall["MISSING_PERCENT"] = (
    ptdemog_missingness_overall["MISSING_ROWS"]
    / len(ptdemog_working)
    * 100
)

ptdemog_missingness_overall = (
    ptdemog_missingness_overall
    .sort_values("MISSING_PERCENT")
    .reset_index(drop=True)
)

# Phase-specific row-level missingness.
phase_missingness_rows = []

for phase, phase_data in ptdemog_working.groupby(
    "PHASE_CLEAN",
    dropna=False,
):
    for column in summary_columns:
        phase_missingness_rows.append(
            {
                "PHASE": phase,
                "VARIABLE": column,
                "TOTAL_ROWS": len(phase_data),
                "NON_MISSING_ROWS": int(
                    phase_data[column].notna().sum()
                ),
                "MISSING_ROWS": int(
                    phase_data[column].isna().sum()
                ),
                "MISSING_PERCENT": (
                    phase_data[column].isna().mean() * 100
                ),
            }
        )

ptdemog_missingness_by_phase = pd.DataFrame(
    phase_missingness_rows
)

display(ptdemog_missingness_overall)

# 33. Summarise participant-level demographic coverage

Because PTDEMOG contains repeated longitudinal records, row-level missingness may overstate or understate practical feature availability.

therefore calculate whether each participant has at least one non-missing value for each cleaned demographic variable.

In [ ]:
participant_coverage_rows = []

for column in summary_columns:
    participant_has_value = (
        ptdemog_working
        .groupby("RID_CLEAN")[column]
        .apply(lambda values: values.notna().any())
    )

    participant_coverage_rows.append(
        {
            "VARIABLE": column,
            "PARTICIPANTS_WITH_VALUE": int(
                participant_has_value.sum()
            ),
            "PARTICIPANTS_WITHOUT_VALUE": int(
                (~participant_has_value).sum()
            ),
            "PARTICIPANT_COVERAGE_PERCENT": (
                participant_has_value.mean() * 100
            ),
        }
    )

ptdemog_participant_coverage = (
    pd.DataFrame(participant_coverage_rows)
    .sort_values(
        "PARTICIPANT_COVERAGE_PERCENT",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(ptdemog_participant_coverage)

# 34. Save the cleaned longitudinal PTDEMOG outputs

save the complete cleaned longitudinal PTDEMOG table together with its quality-control and coverage reports.

All longitudinal records will remain in the cleaned output. Participant-level baseline selection and age calculation will be performed later after the common reference-date and modality-alignment rules have been established.

In [ ]:
# Define output paths.
PTDEMOG_CLEANED_PATH = (
    INTERIM_DIR
    / "ptdemog_cleaned_longitudinal.csv"
)

PTDEMOG_MISSINGNESS_OVERALL_PATH = (
    QC_DIR
    / "ptdemog_missingness_overall.csv"
)

PTDEMOG_MISSINGNESS_BY_PHASE_PATH = (
    QC_DIR
    / "ptdemog_missingness_by_phase.csv"
)

PTDEMOG_PARTICIPANT_COVERAGE_PATH = (
    QC_DIR
    / "ptdemog_participant_coverage.csv"
)

PTDEMOG_PARTICIPANT_CONSISTENCY_PATH = (
    QC_DIR
    / "ptdemog_participant_consistency.csv"
)

PTDEMOG_CORE_CONFLICTS_PATH = (
    QC_DIR
    / "ptdemog_core_conflicts.csv"
)

PTDEMOG_DUPLICATES_PATH = (
    QC_DIR
    / "ptdemog_exact_duplicate_records.csv"
)

PTDEMOG_QC_ERRORS_PATH = (
    QC_DIR
    / "ptdemog_active_qc_error_records.csv"
)

# Save the cleaned longitudinal table.
ptdemog_working.to_csv(
    PTDEMOG_CLEANED_PATH,
    index=False,
)

# Save the QC and coverage outputs.
ptdemog_missingness_overall.to_csv(
    PTDEMOG_MISSINGNESS_OVERALL_PATH,
    index=False,
)

ptdemog_missingness_by_phase.to_csv(
    PTDEMOG_MISSINGNESS_BY_PHASE_PATH,
    index=False,
)

ptdemog_participant_coverage.to_csv(
    PTDEMOG_PARTICIPANT_COVERAGE_PATH,
    index=False,
)

participant_consistency.to_csv(
    PTDEMOG_PARTICIPANT_CONSISTENCY_PATH,
    index=False,
)

ptdemog_conflict_records.to_csv(
    PTDEMOG_CORE_CONFLICTS_PATH,
    index=False,
)

exact_duplicate_records.to_csv(
    PTDEMOG_DUPLICATES_PATH,
    index=False,
)

active_qc_error_records.to_csv(
    PTDEMOG_QC_ERRORS_PATH,
    index=False,
)

print("PTDEMOG preprocessing outputs saved successfully.\n")

print(f"Cleaned longitudinal table:\n{PTDEMOG_CLEANED_PATH}\n")
print(f"Overall missingness:\n{PTDEMOG_MISSINGNESS_OVERALL_PATH}\n")
print(f"Missingness by phase:\n{PTDEMOG_MISSINGNESS_BY_PHASE_PATH}\n")
print(f"Participant coverage:\n{PTDEMOG_PARTICIPANT_COVERAGE_PATH}\n")
print(f"Participant consistency:\n{PTDEMOG_PARTICIPANT_CONSISTENCY_PATH}\n")
print(f"Conflict records:\n{PTDEMOG_CORE_CONFLICTS_PATH}\n")
print(f"Exact duplicate records:\n{PTDEMOG_DUPLICATES_PATH}\n")
print(f"Active QC-error records:\n{PTDEMOG_QC_ERRORS_PATH}")